# Deep learning for change detection on satellite images

In [1]:
# This cell contains the parameters. It should have the tag "parameters"
#Parameters
SRCDIR = "."

In [2]:
# standard imports
import os
import sys
sys.path.append(SRCDIR)  # Will enable local code imports

In [3]:
# library imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Variable
import torchvision.transforms as tr


import numpy as np
from skimage import io
import io as ioo
from IPython import display
import imageio
from IPython.display import Image, display

In [4]:
# local code imports
from fresunet import FresUNet

In [5]:
# text_trap = ioo.StringIO()
# sys.stdout = text_trap
# execute our now mute function

net, net_name = FresUNet(2*3, 2), 'FresUNet3'
net.load_state_dict(torch.load(os.path.join(SRCDIR,'fresunet3_final.pth.tar')))
# now restore 
#stdout function
# sys.stdout = sys.__stdout__



<All keys matched successfully>

In [23]:
import os
import glob
import rasterio
import numpy as np
from PIL import Image

def from_S2_to_RGB(input_folder, output_folder="./data", red_b=3, green_b=2, blue_b=1):
    """
    Reads Sentinel-2 *.tif images, converts them to RGB 8-bit PNGs using 
    a 0.3 contrast stretch and NaN handling, and saves them.
    
    Parameters:
    - input_folder (str): Path to the folder containing the raw .tif images.
    - output_folder (str): Path to save the converted .png images.
    - red_b, green_b, blue_b (int): Rasterio 1-based band indices for Red, Green, Blue.
    """
    # Create the output directory if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Find all files matching the pattern *.tif
    search_pattern = os.path.join(input_folder, "*.tif")
    tif_files = glob.glob(search_pattern)
    
    if not tif_files:
        print(f"No files matching '*.tif' found in {input_folder}")
        return []

    for tif_path in tif_files:
        print(f"Processing: {os.path.basename(tif_path)}")
        
        # 1. Read the raw TIFF file
        with rasterio.open(tif_path) as src:
            # rasterio uses 1-based indexing for bands
            r = src.read(red_b).astype(np.float32)
            g = src.read(green_b).astype(np.float32)
            b = src.read(blue_b).astype(np.float32)
            
        # 2. Stack into an RGB array (Height, Width, Channels)
        rgb = np.dstack((r, g, b))
        
        # 3. Apply logic from tiff_viewer.py
        # Handle -10000 NoData values by converting to NaN first
        rgb[rgb == -10000] = np.nan
        
        # Apply Sentinel-2 scaling factor
        rgb = rgb / 10000.0
        
        # Replace NaNs with 0.0 (black pixels) to prevent PIL/Matplotlib errors
        rgb = np.nan_to_num(rgb, nan=0.0)
        
        # 4. Apply the contrast stretch (/ 0.3) and clip to [0, 1]
        rgb_clipped = np.clip(rgb / 0.3, 0, 1)
        
        # 5. Scale to [0, 255] for 8-bit PNG creation
        rgb_8bit = (rgb_clipped * 255).astype(np.uint8)
        
        # 6. Create and save the PNG
        img = Image.fromarray(rgb_8bit)
        
        base_filename = os.path.basename(tif_path)
        png_filename = base_filename.replace('.tif', '.png')
        out_path = os.path.join(output_folder, png_filename)
        
        img.save(out_path)

# --- Example Usage ---
run_name = "run_009"
tif_folder = f"../../all/{run_name}/predictions/30SWH_24/hr_mae_CUSTOM_FORECAST_50_318.0"

png_folder = f"./data/{run_name}/inputs"
from_S2_to_RGB(input_folder=tif_folder, output_folder=png_folder)

Processing: 107_2022-04-18_hr_mae_CUSTOM_FORECAST_50_318.0_pred.tif
Processing: 107_2022-04-18_hr_mae_CUSTOM_FORECAST_50_318.0_ref.tif
Processing: 127_2022-05-08_hr_mae_CUSTOM_FORECAST_50_318.0_pred.tif
Processing: 127_2022-05-08_hr_mae_CUSTOM_FORECAST_50_318.0_ref.tif
Processing: 132_2022-05-13_hr_mae_CUSTOM_FORECAST_50_318.0_pred.tif
Processing: 132_2022-05-13_hr_mae_CUSTOM_FORECAST_50_318.0_ref.tif
Processing: 137_2022-05-18_hr_mae_CUSTOM_FORECAST_50_318.0_pred.tif
Processing: 137_2022-05-18_hr_mae_CUSTOM_FORECAST_50_318.0_ref.tif
Processing: 142_2022-05-23_hr_mae_CUSTOM_FORECAST_50_318.0_pred.tif
Processing: 142_2022-05-23_hr_mae_CUSTOM_FORECAST_50_318.0_ref.tif
Processing: 147_2022-05-28_hr_mae_CUSTOM_FORECAST_50_318.0_pred.tif
Processing: 147_2022-05-28_hr_mae_CUSTOM_FORECAST_50_318.0_ref.tif
Processing: 152_2022-06-02_hr_mae_CUSTOM_FORECAST_50_318.0_pred.tif
Processing: 152_2022-06-02_hr_mae_CUSTOM_FORECAST_50_318.0_ref.tif
Processing: 157_2022-06-07_hr_mae_CUSTOM_FORECAST_50_31

In [24]:
def LoadImages(name1,name2):
    I1 = io.imread(name1).astype('float')
    I1 = (I1 - I1.mean()) / I1.std()

    imageio.imsave("output_0.png",simple_equalization_8bit(I1))
    I2 = io.imread(name2).astype('float')
    I2 = (I2 - I2.mean()) / I2.std()

    imageio.imsave("output_1.png",simple_equalization_8bit(I2))
    
    # crop if necessary
    s1 = I1.shape
    s2 = I2.shape
    I2 = np.pad(I2,((0, s1[0] - s2[0]), (0, s1[1] - s2[1]), (0,0)),'edge')
    
    im1 = reshape_for_torch(I1)
    im2 = reshape_for_torch(I2)
    
    return im1,im2

def reshape_for_torch(I):
    """Transpose image for PyTorch coordinates."""
    out = I.transpose((2, 0, 1))
    return torch.from_numpy(out)


def simple_equalization_8bit(im, percentiles=5):
    """
    Simple 8-bit requantization by linear stretching.

    Args:
        im (np.array): image to requantize
        percentiles (int): percentage of the darkest and brightest pixels to saturate

    Returns:
        numpy array with the quantized uint8 image
    """
    import numpy as np
    mi, ma = np.percentile(im[np.isfinite(im)], (percentiles, 100 - percentiles))
    im = np.clip(im, mi, ma)
    im = (im - mi) / (ma - mi) * 255   # scale
    return im.astype(np.uint8)


def compute_map(name1,name2):
    I1, I2 = LoadImages(name1,name2)
    I1 = Variable(torch.unsqueeze(I1, 0).float()) #.cuda()
    I2 = Variable(torch.unsqueeze(I2, 0).float()) #.cuda()
    out = net(I1, I2)
    _, predicted = torch.max(out.data, 1)
    return predicted


In [29]:
import os
import glob
import imageio

search_pattern = os.path.join(png_folder, "*_ref.png")
ref_files = glob.glob(search_pattern)

# --- NEW SORTING LOGIC ---
def extract_numerical_prefix(filepath):
    """
    Extracts the first number from the filename (e.g., '47' from '47_2022-02-17...') 
    and returns it as an integer so Python sorts it mathematically.
    """
    filename = os.path.basename(filepath)
    prefix = filename.split('_')[0]
    return int(prefix)

# Sort the list in-place using our custom numerical key
ref_files.sort(key=extract_numerical_prefix)
# -------------------------

print(f"Found {len(ref_files)} reference images. Looking for matches...")

os.makedirs(f"./data/{run_name}/outputs/prediction/", exist_ok=True)
os.makedirs(f"./data/{run_name}/outputs/temporal_difference/", exist_ok=True)

for i, name1 in enumerate(ref_files):
    # Construct the expected path for the corresponding prediction image
    name2 = name1.replace('_ref.png', '_pred.png')

    print("\n")
    print(f"Ref (t):  {os.path.basename(name1)}")
    print(f"Pred (t): {os.path.basename(name2)}")

    p = compute_map(name1, name2)
    pred_output_path = f"./data/{run_name}/outputs/prediction/{os.path.basename(name1.replace('_ref.png', '.png'))}"
    imageio.imsave(pred_output_path, (255 * p[0,:,:]).numpy().astype(np.uint8))

    # Calculate temporal difference only if there is a previous image
    if i > 0:
        name3 = ref_files[i-1]

        print(f"Ref (t-1): {os.path.basename(name3)}")

        p_td = compute_map(name1, name3)
        td_output_path = f"./data/{run_name}/outputs/temporal_difference/{os.path.basename(name1.replace('_ref.png', '.png'))}"
        imageio.imsave(td_output_path, (255 * p_td[0,:,:]).numpy().astype(np.uint8))

Found 41 reference images. Looking for matches...


Ref (t):  32_2022-02-02_hr_mae_CUSTOM_FORECAST_50_318.0_ref.png
Pred (t): 32_2022-02-02_hr_mae_CUSTOM_FORECAST_50_318.0_pred.png


Ref (t):  37_2022-02-07_hr_mae_CUSTOM_FORECAST_50_318.0_ref.png
Pred (t): 37_2022-02-07_hr_mae_CUSTOM_FORECAST_50_318.0_pred.png
Ref (t-1): 32_2022-02-02_hr_mae_CUSTOM_FORECAST_50_318.0_ref.png


Ref (t):  42_2022-02-12_hr_mae_CUSTOM_FORECAST_50_318.0_ref.png
Pred (t): 42_2022-02-12_hr_mae_CUSTOM_FORECAST_50_318.0_pred.png
Ref (t-1): 37_2022-02-07_hr_mae_CUSTOM_FORECAST_50_318.0_ref.png


Ref (t):  47_2022-02-17_hr_mae_CUSTOM_FORECAST_50_318.0_ref.png
Pred (t): 47_2022-02-17_hr_mae_CUSTOM_FORECAST_50_318.0_pred.png
Ref (t-1): 42_2022-02-12_hr_mae_CUSTOM_FORECAST_50_318.0_ref.png


Ref (t):  52_2022-02-22_hr_mae_CUSTOM_FORECAST_50_318.0_ref.png
Pred (t): 52_2022-02-22_hr_mae_CUSTOM_FORECAST_50_318.0_pred.png
Ref (t-1): 47_2022-02-17_hr_mae_CUSTOM_FORECAST_50_318.0_ref.png


Ref (t):  67_2022-03-09_hr_mae_CU